# 03 Feature Extraction

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`HGB_N5_measure_C2.ipynb`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`more.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/more.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`n8.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/n8.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)
- [`run_all.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/run_all.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '60e421dddaee847a5d7701924281f7d202d24d9b'}, '05_model_optimize/HGB/Normal/scripts/more.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b4c1738e7f1b48c277bf03b927f4208c523fb2b6'}, '05_model_optimize/HGB/Normal/scripts/n8.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2d7f307efe8b54fad07c4e705b7d50e08181f0f7'}, '05_model_optimize/HGB/Normal/scripts/run_all.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '4e9ff3ebccf8fa3a5d92eb17de6de07f09ea8558'}}

## HGB_N5_measure_C2

원본: [05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb)  
Git blob: `60e421dddaee847a5d7701924281f7d202d24d9b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

원본 Notebook의 Markdown 및 실행 코드 셀을 순서대로 보존한다. 기존 출력·이미지는 복제하지 않는다. 원본 Colab 환경/실행 순서는 원본 링크 참조.

### 원본 셀 0

# HGB 보충 측정(N5) + C2 Combined

| 단계 | 내용 | 시간 |
|---|---|---|
| S | N5 재학습 → **학습곡선**(반복별 손실·정확도), **예측 시간**, **Macro Precision·Recall** | 약 45분 |
| C2 | N5 구성(픽셀+투영+HOG)을 Combined에 적용, 작성자 보류 4-fold OOF | 약 3시간 |

결과: 드라이브 `HGB_results/HGB_results_v4.csv`, 그림 `S_learning_curve.png`, `HGB_confusion_C2.png`

**실행법:** 0번 셀 실행(세션 다운 = 정상) → 1번 셀 클릭 → 런타임 → 이후 셀 실행

### 원본 셀 1

## 0. scikit-learn 1.8.0 설치 (세션이 재시작돼요. 정상)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb#cell2
!pip install -q scikit-learn==1.8.0
import os; os.kill(os.getpid(), 9)

### 원본 셀 3

## 1. 준비

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb#cell4
!test -d /content/Machine_Learning || git clone --depth 1 https://github.com/ryu980920/Machine_Learning.git /content/Machine_Learning
from google.colab import drive
drive.mount('/content/drive')
import os, sys, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
OUT = '/content/drive/MyDrive/HGB_results/'; os.makedirs(OUT, exist_ok=True)
REPO = Path('/content/Machine_Learning')
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Combined/scripts'))
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from skimage.feature import hog

off = load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz')
wr  = load_writers(REPO/'02_data/Hand_made')
WRITERS = list(L.WRITERS); LABELS = '0123456789ABCDEF'
ytr, yva = off['y_train'], off['y_val']
yhm = np.concatenate([wr[w]['y'] for w in WRITERS])
whm = np.concatenate([[w]*len(wr[w]['y']) for w in WRITERS])

Xtr_aug, tw_com, _, _ = L.prepareraw(off, wr, 'scale30_stroke28')   # N3/N5 전처리
Xva_com = process(off['X_val'], 'com', 20)
Xhm_com = np.vstack([tw_com[w] for w in WRITERS])

def shape_feats(X):
    X = X.reshape(-1, 28, 28).astype(np.float32)
    proj = np.hstack([X.sum(1), X.sum(2)]) / 255.0
    H = np.stack([hog(im, orientations=9, pixels_per_cell=(7,7), cells_per_block=(2,2)) for im in X])
    return np.hstack([proj, H]).astype(np.float32)

t = time.time()
Ftr = np.hstack([Xtr_aug, shape_feats(Xtr_aug)])
Fva = np.hstack([Xva_com, shape_feats(Xva_com)])
Fhm_map = {w: np.hstack([tw_com[w], shape_feats(tw_com[w])]) for w in WRITERS}
Fhm = np.vstack([Fhm_map[w] for w in WRITERS])
feat_s = round(time.time() - t)
print('특징 추출', feat_s, '초', Ftr.shape, Fva.shape, Fhm.shape)   # (76800,1164) (19200,1164) (2560,1164)

### 원본 셀 5

## 2. 공통 함수

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb#cell6
PAIRS = [('D','0'), ('0','D'), ('B','8'), ('8','B'), ('5','F'), ('7','9')]
CSV = OUT + 'HGB_results_v4.csv'
results = []

def scores(y, p, prefix):
    return {f'{prefix}_acc': accuracy_score(y, p),
            f'{prefix}_precision': precision_score(y, p, average='macro', zero_division=0),
            f'{prefix}_recall': recall_score(y, p, average='macro', zero_division=0),
            f'{prefix}_f1': f1_score(y, p, average='macro')}

def hm_detail(p):
    isd = yhm < 10
    r = dict(hm_digit=accuracy_score(yhm[isd], p[isd]), hm_letter=accuracy_score(yhm[~isd], p[~isd]))
    for w in WRITERS: r[f'w{w}'] = accuracy_score(yhm[whm==w], p[whm==w])
    for a, b in PAIRS: r[f'{a}->{b}'] = int(np.sum((yhm==LABELS.index(a)) & (p==LABELS.index(b))))
    return r

def save(row):
    results.append(row); pd.DataFrame(results).to_csv(CSV, index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})

def hgb():
    return HistGradientBoostingClassifier(max_iter=300, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=10, random_state=42)

### 원본 셀 7

## S. N5 보충 측정 — 학습 시간·예측 시간·Precision·Recall·학습곡선

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb#cell8
m5 = hgb()
t = time.time(); m5.fit(Ftr, ytr); fit_s = time.time() - t

t = time.time(); pv = m5.predict(Fva); pred_va_s = time.time() - t
t = time.time(); ph = m5.predict(Fhm); pred_hm_s = time.time() - t

save(dict(name='S_N5_재현+보충측정', **scores(yva, pv, 'val'), **scores(yhm, ph, 'hm'), **hm_detail(ph),
          fit_s=round(fit_s), iters=m5.n_iter_, feature_s=feat_s,
          predict_val_s=round(pred_va_s, 2), predict_hm_s=round(pred_hm_s, 2),
          predict_ms_per_img=round(1000*pred_va_s/len(yva), 4)))

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb#cell9
# 학습곡선: 반복(트리 개수)마다 ① 내부 학습/검증 손실 ② 공식 Validation·손글씨 정확도
it = np.arange(len(m5.train_score_))
acc_va = [accuracy_score(yva, p) for p in m5.staged_predict(Fva)]
acc_hm = [accuracy_score(yhm, p) for p in m5.staged_predict(Fhm)]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(it, -m5.train_score_, label='Train loss (internal subsample)')
ax[0].plot(it, -m5.validation_score_, label='Validation loss (internal 10%)')
ax[0].set_xlabel('Boosting iteration (number of trees per class)'); ax[0].set_ylabel('Log loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(np.arange(1, len(acc_va)+1), acc_va, label='Official Validation 19,200')
ax[1].plot(np.arange(1, len(acc_hm)+1), acc_hm, label='Hand-made 2,560 (dev)')
ax[1].set_xlabel('Boosting iteration'); ax[1].set_ylabel('Accuracy'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle(f'N5 HGB learning curve (early stopped at {m5.n_iter_})'); plt.tight_layout()
plt.savefig(OUT+'S_learning_curve.png', dpi=130); plt.show()

lc = pd.DataFrame({'iteration': it, 'train_loss': -m5.train_score_, 'val_loss': -m5.validation_score_})
lc['official_val_acc'] = [np.nan] + acc_va[:len(it)-1]; lc['handmade_acc'] = [np.nan] + acc_hm[:len(it)-1]
lc.to_csv(OUT+'S_learning_curve.csv', index=False)
i_best = int(np.argmin(-m5.validation_score_))
print(f'내부 검증 손실 최소: {i_best}회 ({-m5.validation_score_[i_best]:.4f}), 마지막 {it[-1]}회 ({-m5.validation_score_[-1]:.4f})')
print(f'학습 손실 마지막: {-m5.train_score_[-1]:.4f}')

### 원본 셀 10

## C2. Combined — N5 구성 + 다른 작성자 3명 1,920장 추가, 작성자 보류 4-fold OOF

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/colab/HGB_N5_measure_C2.ipynb#cell11
oof = np.zeros(len(yhm), dtype=int); fold = []
for w in WRITERS:
    others = [o for o in WRITERS if o != w]
    X = np.vstack([Ftr] + [Fhm_map[o] for o in others])
    y = np.concatenate([ytr] + [wr[o]['y'] for o in others])
    m = hgb()
    t = time.time(); m.fit(X, y); fs = time.time() - t
    t = time.time(); oof[whm == w] = m.predict(Fhm_map[w]); ps = time.time() - t
    a = accuracy_score(yhm[whm==w], oof[whm==w])
    fold.append(dict(writer=w, train_n=len(y), acc=a, fit_s=round(fs), iters=m.n_iter_, predict_s=round(ps, 2)))
    print(fold[-1])
    pd.DataFrame(fold).to_csv(OUT+'C2_folds.csv', index=False)

save(dict(name='C2_Combined_N5구성', **scores(yhm, oof, 'hm'), **hm_detail(oof),
          fit_s=round(np.mean([f['fit_s'] for f in fold])), iters=round(np.mean([f['iters'] for f in fold]), 1)))

from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(yhm, oof, labels=range(16), display_labels=list(LABELS),
                                        normalize='true', values_format='.2f', ax=ax, colorbar=False)
ax.set_title('C2 Combined (N5 features) writer-held-out OOF')
plt.tight_layout(); plt.savefig(OUT+'HGB_confusion_C2.png', dpi=130); plt.show()
pd.DataFrame(results)

## more

원본: [05_model_optimize/HGB/Normal/scripts/more.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/more.py)  
Git blob: `b4c1738e7f1b48c277bf03b927f4208c523fb2b6`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/more.py
from pathlib import Path
import sys,time,json,numpy as np
stage=sys.argv[1]; sys.argv=['x','none']; exec(open(Path(__file__).resolve().parent/'run_all.py').read().split("stage=sys.argv[1]")[0])
from sklearn.ensemble import HistGradientBoostingClassifier
def H(seed): return HistGradientBoostingClassifier(max_iter=300,early_stopping=True,validation_fraction=0.1,n_iter_no_change=10,random_state=seed)
Ftr=np.load(OUT/'Ftr.npy'); Fva=np.load(OUT/'Fva.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}; Fhm=np.vstack([Fm[w] for w in W])
cols={'pix':slice(0,784),'proj':slice(784,840),'hog':slice(840,1164)}
def sel(X,parts): return np.hstack([X[:,cols[p]] for p in parts])
kind,seed=stage.split('_'); seed=int(seed)
if kind=='n3': A,B,C=Xtr,Xva,Xhm
elif kind=='n5': A,B,C=Ftr,Fva,Fhm
elif kind=='proj': A,B,C=[sel(X,['pix','proj']) for X in (Ftr,Fva,Fhm)]
elif kind=='hog': A,B,C=[sel(X,['pix','hog']) for X in (Ftr,Fva,Fhm)]
m=H(seed); t=time.time(); m.fit(A,ytr); fs=time.time()-t
pv=m.predict(B); ph=m.predict(C)
r=dict(name=f'{kind}_s{seed}',n_feat=A.shape[1],**sc(yva,pv,'val'),**sc(yhm,ph,'hm'),**det(ph),fit_s=round(fs),iters=int(m.n_iter_))
json.dump(r,open(OUT/f'more_{stage}.json','w')); np.save(OUT/f'pred_{stage}.npy',ph); print(r,flush=True)


## n8

원본: [05_model_optimize/HGB/Normal/scripts/n8.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/n8.py)  
Git blob: `2d7f307efe8b54fad07c4e705b7d50e08181f0f7`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/n8.py
from pathlib import Path
import sys,time,json,numpy as np
seed=int(sys.argv[1]); sys.argv=['x','none']; exec(open(Path(__file__).resolve().parent/'run_all.py').read().split("stage=sys.argv[1]")[0])
from sklearn.ensemble import HistGradientBoostingClassifier
Ftr=np.load(OUT/'Ftr.npy'); Fva=np.load(OUT/'Fva.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}; Fhm=np.vstack([Fm[w] for w in W])
def build(F):
    pr=F[:,784:840]; tot=pr[:,:28].sum(1,keepdims=True)+1e-6
    return np.hstack([F[:,:784],F[:,840:1164],pr/tot]).astype(np.float32)
A,B,C=build(Ftr),build(Fva),build(Fhm)
m=HistGradientBoostingClassifier(max_iter=300,early_stopping=True,validation_fraction=0.1,n_iter_no_change=10,random_state=seed)
t=time.time(); m.fit(A,ytr); fs=time.time()-t
pv=m.predict(B); ph=m.predict(C)
r=dict(name=f'N8_s{seed}',**sc(yva,pv,'val'),**sc(yhm,ph,'hm'),**det(ph),fit_s=round(fs),iters=int(m.n_iter_))
json.dump(r,open(OUT/f'n8_s{seed}.json','w')); np.save(OUT/f'pred_n8_s{seed}.npy',ph); print(r,flush=True)


## run_all

원본: [05_model_optimize/HGB/Normal/scripts/run_all.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/scripts/run_all.py)  
Git blob: `4e9ff3ebccf8fa3a5d92eb17de6de07f09ea8558`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Normal/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Normal/scripts/run_all.py
from pathlib import Path
import sys, time, json, numpy as np, pandas as pd
from pathlib import Path
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
REPO=Path(__file__).resolve().parents[4]; OUT=Path(__file__).resolve().parent/'out'; OUT.mkdir(exist_ok=True)
sys.path.insert(0,str(REPO/'05_model_optimize/MLP/Combined/scripts')); sys.path.insert(0,str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,ConfusionMatrixDisplay
from skimage.feature import hog
import sklearn; print('sklearn',sklearn.__version__,flush=True)
off=load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz'); wr=load_writers(REPO/'02_data/Hand_made')
W=list(L.WRITERS); LAB='0123456789ABCDEF'; ytr,yva=off['y_train'],off['y_val']
yhm=np.concatenate([wr[w]['y'] for w in W]); whm=np.concatenate([[w]*640 for w in W])
Xtr,tw,_,_=L.prepareraw(off,wr,'scale30_stroke28'); Xva=process(off['X_val'],'com',20); Xhm=np.vstack([tw[w] for w in W])
PAIRS=[('D','0'),('0','D'),('B','8'),('8','B'),('5','F'),('7','9')]
def hgb(**k): return HistGradientBoostingClassifier(max_iter=300,early_stopping=True,validation_fraction=0.1,n_iter_no_change=10,random_state=42,**k)
def sc(y,p,pre): return {f'{pre}_acc':accuracy_score(y,p),f'{pre}_precision':precision_score(y,p,average='macro',zero_division=0),f'{pre}_recall':recall_score(y,p,average='macro',zero_division=0),f'{pre}_f1':f1_score(y,p,average='macro')}
def det(p):
    d=yhm<10; r=dict(hm_digit=accuracy_score(yhm[d],p[d]),hm_letter=accuracy_score(yhm[~d],p[~d]))
    for w in W: r[f'w{w}']=accuracy_score(yhm[whm==w],p[whm==w])
    for a,b in PAIRS: r[f'{a}->{b}']=int(np.sum((yhm==LAB.index(a))&(p==LAB.index(b))))
    return r
res=[]
def save(r): res.append(r); pd.DataFrame(res).to_csv(OUT/'results.csv',index=False); print(r,flush=True)
stage=sys.argv[1]
if stage=='n3':
    m=hgb(); t=time.time(); m.fit(Xtr,ytr); fs=time.time()-t
    pv=m.predict(Xva); ph=m.predict(Xhm)
    save(dict(name='N3',**sc(yva,pv,'val'),**sc(yhm,ph,'hm'),**det(ph),fit_s=round(fs),iters=m.n_iter_))
    np.save(OUT/'ph3.npy',ph)
    def gm(cls=None):
        g=np.zeros(784)
        for it in m._predictors:
            for tr in (it if cls is None else [it[cls]]):
                n=tr.nodes[tr.nodes['is_leaf']==0]; np.add.at(g,n['feature_idx'],n['gain'])
        return g.reshape(28,28)
    maps={'All':gm()}|{c:gm(LAB.index(c)) for c in 'D0B8'}
    np.savez(OUT/'gain.npz',**maps)
    fig,ax=plt.subplots(1,5,figsize=(20,4.5))
    for a,(k,g) in zip(ax,maps.items()): a.imshow(g,cmap='hot'); a.set_title(f'{k} trees'); a.axis('off')
    plt.tight_layout(); plt.savefig(OUT/'D1_gain_heatmap.png',dpi=130)
    st={}
    for k,g in maps.items():
        s=np.sort(g.ravel())[::-1]; st[k]=dict(used=int((g>0).sum()),top20=float(s[:20].sum()/s.sum()))
    json.dump(st,open(OUT/'D1_stats.json','w')); print(st,flush=True)
if stage=='n5':
    def sf(X):
        X=X.reshape(-1,28,28).astype(np.float32); proj=np.hstack([X.sum(1),X.sum(2)])/255.
        return np.hstack([proj,np.stack([hog(i,orientations=9,pixels_per_cell=(7,7),cells_per_block=(2,2)) for i in X])]).astype(np.float32)
    t=time.time(); Ftr=np.hstack([Xtr,sf(Xtr)]); Fva=np.hstack([Xva,sf(Xva)]); Fm={w:np.hstack([tw[w],sf(tw[w])]) for w in W}; Fhm=np.vstack([Fm[w] for w in W]); fe=time.time()-t
    np.save(OUT/'Ftr.npy',Ftr); np.save(OUT/'Fva.npy',Fva); np.savez(OUT/'Fm.npz',**{str(k):v for k,v in Fm.items()})
    m=hgb(); t=time.time(); m.fit(Ftr,ytr); fs=time.time()-t
    t=time.time(); pv=m.predict(Fva); pvs=time.time()-t; t=time.time(); ph=m.predict(Fhm); phs=time.time()-t
    t=time.time(); m.predict(Fva[:1]); one=time.time()-t
    save(dict(name='N5',**sc(yva,pv,'val'),**sc(yhm,ph,'hm'),**det(ph),fit_s=round(fs),iters=m.n_iter_,feature_s=round(fe,1),
              predict_val_s=round(pvs,3),predict_hm_s=round(phs,3),ms_per_img=round(1000*pvs/len(yva),4),single_img_ms=round(1000*one,2)))
    np.save(OUT/'ph5.npy',ph)
    it=np.arange(len(m.train_score_))
    av=[accuracy_score(yva,p) for p in m.staged_predict(Fva)]; ah=[accuracy_score(yhm,p) for p in m.staged_predict(Fhm)]
    lc=pd.DataFrame({'iteration':it,'train_loss':-m.train_score_,'val_loss':-m.validation_score_,'official_val_acc':[np.nan]+av[:len(it)-1],'handmade_acc':[np.nan]+ah[:len(it)-1]})
    lc.to_csv(OUT/'learning_curve.csv',index=False); print(lc.iloc[[1,10,50,100,-1]],flush=True)
if stage.startswith('c2_'):
    w=int(stage[3:]); Ftr=np.load(OUT/'Ftr.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}
    o=[x for x in W if x!=w]; X=np.vstack([Ftr]+[Fm[x] for x in o]); y=np.concatenate([ytr]+[wr[x]['y'] for x in o])
    m=hgb(); t=time.time(); m.fit(X,y); fs=time.time()-t; t=time.time(); p=m.predict(Fm[w]); ps=time.time()-t
    np.save(OUT/f'c2_{w}.npy',p); json.dump(dict(writer=w,train_n=len(y),acc=accuracy_score(wr[w]['y'],p),fit_s=round(fs),iters=int(m.n_iter_),predict_s=round(ps,3)),open(OUT/f'c2_{w}.json','w'))
    print('fold',w,accuracy_score(wr[w]['y'],p),round(fs),m.n_iter_,flush=True)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")